## time列の追加

In [1]:
import pandas as pd
import os

# データ読み込み
df = pd.read_csv("../data/raw/test.csv", encoding='cp932')

# 各樹種ごとにsample numberを1始まりで振り直したtime列を追加
df['time'] = df.groupby('species number')['sample number'].rank(method='first').astype(int)

# time列をsample number, species numberの直後に移動
cols = df.columns.tolist()
cols.insert(2, cols.pop(cols.index('time')))
df = df[cols]

# 保存
os.makedirs("../data/middle", exist_ok=True)
save_path = "../data/middle/test_with_time.csv"
df.to_csv(save_path, index=False, encoding='cp932')

print(f"shape: {df.shape}")
print(f"保存完了: {save_path}")
print("\ntime範囲（樹種別）:")
print(df.groupby('species number')['time'].agg(['min', 'max', 'count']))
df.head()


shape: (550, 1559)
保存完了: ../data/middle/test_with_time.csv

time範囲（樹種別）:
                min  max  count
species number                 
2                 1   96     96
6                 1   63     63
7                 1  192    192
9                 1   45     45
10                1  101    101
18                1   53     53


,sample number,species number,time,樹種,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,...,4034.53536,4030.67826,4026.82115,4022.96404,4019.10693,4015.24982,4011.39271,4007.5356,4003.6785,3999.82139
0,95,2,1,クスノキ,0.32877,0.32855,0.32823,0.32792,0.32759,0.32729,...,1.35443,1.36345,1.36470,1.36110,1.35863,1.35918,1.36074,1.35224,1.33495,1.32482
1,96,2,2,クスノキ,0.30061,0.30006,0.29978,0.29984,0.29977,0.29945,...,1.12196,1.12506,1.12357,1.11911,1.11655,1.11955,1.12671,1.13182,1.13070,1.12377
2,97,2,3,クスノキ,0.26048,0.26033,0.26000,0.25970,0.25952,0.25945,...,0.98370,0.98904,0.99325,0.99610,0.99886,0.99919,0.99367,0.98723,0.98868,0.99461
3,98,2,4,クスノキ,0.23210,0.23188,0.23167,0.23158,0.23142,0.23122,...,0.93830,0.94524,0.94897,0.95018,0.95243,0.95574,0.95703,0.95530,0.95444,0.95832
4,99,2,5,クスノキ,0.19725,0.19704,0.19687,0.19676,0.19660,0.19644,...,0.87793,0.88175,0.88647,0.89267,0.89790,0.90191,0.90292,0.90119,0.89835,0.89623


## binごとにスペクトルを分けるデータ

In [4]:
import pandas as pd
import numpy as np
import os

# ===== パラメータ =====
bin_width = 1000   # ビン幅 (cm⁻¹)
# ====================

# データ読み込み（time列付きtestデータを使用）
df = pd.read_csv("../data/middle/test_with_time.csv", encoding='cp932')

meta_cols = ['sample number', 'species number', 'time', '樹種']
spec_cols = [c for c in df.columns if c not in meta_cols]
spec_cols_float = [(c, float(c)) for c in spec_cols if float(c) >= 4000.0]

# ビニング
bins = np.arange(4000, 10000 + bin_width, bin_width)

bin_col_map = {}
for i in range(len(bins) - 1):
    low, high = bins[i], bins[i + 1]
    label = f"{int(low)}-{int(high)}"
    matched = [c for c, v in spec_cols_float if low <= v < high]
    if matched:
        bin_col_map[label] = matched

df_out = df[meta_cols].copy()
for label, cols in bin_col_map.items():
    df_out[label] = df[cols].mean(axis=1)

# 保存
os.makedirs("../data/middle", exist_ok=True)
save_path = f"../data/middle/test_spectrum_bin{bin_width}.csv"
df_out.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"ビン幅: {bin_width} cm⁻¹ | ビン数: {len(bin_col_map)}")
print(f"shape: {df_out.shape}")
print(f"保存完了: {save_path}")
df_out.head()


ビン幅: 1000 cm⁻¹ | ビン数: 6
shape: (550, 10)
保存完了: ../data/middle/test_spectrum_bin1000.csv


,sample number,species number,time,樹種,4000-5000,5000-6000,6000-7000,7000-8000,8000-9000,9000-10000
0,95,2,1,クスノキ,1.187713,0.971101,0.916311,0.589577,0.403026,0.312028
1,96,2,2,クスノキ,0.971915,0.801765,0.761799,0.512401,0.366665,0.286676
2,97,2,3,クスノキ,0.834732,0.685234,0.641281,0.436383,0.317529,0.249275
3,98,2,4,クスノキ,0.776395,0.631283,0.582107,0.392717,0.284921,0.222423
4,99,2,5,クスノキ,0.709768,0.570277,0.516420,0.342644,0.246027,0.189577
